# ComfyUI su GPU T4 (Kaggle)

Versione Kaggle di `comfyui_t4.ipynb`. Sul PC non cambia niente: `genera.py` si usa con il link della cella 5.

**Prima di iniziare**, nel pannello a destra del notebook (**Settings**):
- **Accelerator** → **GPU T4 x2**
- **Internet** → **On** (la prima volta Kaggle chiede di verificare il numero di telefono)

Poi esegui le celle **in ordine**:
1. **Controllo**: verifica GPU e Internet.
2. **Installazione**: ComfyUI + ComfyUI-Manager.
3. **Modelli**: Z-Image Turbo (circa 21 GB, qualche minuto).
4. **Output**: niente Drive su Kaggle, le immagini le scarica `genera.py` sul PC.
5. **Avvio**: stampa il link `https://....trycloudflare.com`.

Lascia l'ultima cella in esecuzione e la scheda aperta finché lavori.
Quando hai finito: **Run → Stop session** (altrimenti consumi le ore GPU settimanali).

In [ ]:
# === 1. CONTROLLO GPU E INTERNET ===
import os, socket, subprocess

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0 or 'T4' not in gpu.stdout:
    raise RuntimeError('Nessuna GPU T4: in Settings scegli Accelerator -> GPU T4 x2 e riavvia la sessione.')

try:
    socket.create_connection(('huggingface.co', 443), timeout=10).close()
    print('Internet: OK')
except OSError:
    raise RuntimeError('Internet spento: in Settings attiva Internet -> On (serve il telefono verificato).')

In [ ]:
# === 2. INSTALLAZIONE COMFYUI + MANAGER ===
# Su Kaggle /kaggle/working ha un limite di 20 GB: ComfyUI e i modelli vanno in /tmp.
COMFY = '/tmp/ComfyUI'
MANAGER = f'{COMFY}/custom_nodes/ComfyUI-Manager'

if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
else:
    !git -C {COMFY} pull

if not os.path.exists(MANAGER):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager {MANAGER}

!pip install -q -r {COMFY}/requirements.txt
!pip install -q -r {MANAGER}/requirements.txt
print('Installazione completata.')

In [ ]:
# === 3. MODELLI ===
# Togli il # davanti ai modelli che vuoi scaricare.
# I modelli vanno riscaricati a ogni nuova sessione.
MODELLI = {
    # Z-Image Turbo (~21 GB in tutto, 8 passi): personaggi del cartone
    'diffusion_models/z_image_turbo_bf16.safetensors':
        'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors',
    'text_encoders/qwen_3_4b.safetensors':
        'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors',
    'vae/ae.safetensors':
        'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors',

    # Stable Diffusion 1.5 (~2 GB), per workflows/immagine_sd15.json
    # 'checkpoints/v1-5-pruned-emaonly-fp16.safetensors':
    #     'https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors',
}

# wget -c riprende i download interrotti e salta quelli già completi.
for percorso, url in MODELLI.items():
    dest = f'{COMFY}/models/{percorso}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print('Modello:', percorso)
    !wget -q --show-progress -c -O '{dest}' '{url}'
!df -h /tmp | tail -1

In [ ]:
# === 4. CARTELLA DI OUTPUT ===
# Su Kaggle non c'è Google Drive: le immagini restano qui finché dura la sessione,
# ma genera.py le scarica comunque sul PC.
OUTPUT_DIR = f'{COMFY}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Le immagini verranno salvate in:', OUTPUT_DIR)

In [ ]:
# === 5. AVVIO COMFYUI + LINK PUBBLICO ===
import re, time, threading

# Chiude eventuali avvii precedenti (se riesegui la cella)
subprocess.run(['pkill', '-f', 'main.py --listen'])
subprocess.run(['pkill', '-f', 'cloudflared'])

# Installa cloudflared (crea il link pubblico verso ComfyUI)
CLOUDFLARED = '/tmp/cloudflared'
if not os.path.exists(CLOUDFLARED):
    !wget -q -O {CLOUDFLARED} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CLOUDFLARED}

# Avvia ComfyUI in background sulla prima T4 (log in /tmp/comfyui.log)
log = open('/tmp/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     '--output-directory', OUTPUT_DIR, '--cuda-device', '0'],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)

print('Avvio ComfyUI (la prima volta può richiedere qualche minuto)...')
while True:
    try:
        socket.create_connection(('127.0.0.1', 8188), timeout=1).close()
        break
    except OSError:
        if comfy.poll() is not None:
            !tail -n 40 /tmp/comfyui.log
            raise RuntimeError('ComfyUI si è chiuso: vedi il log qui sopra.')
        time.sleep(2)
print('ComfyUI pronto. Apro il tunnel...')

tunnel = subprocess.Popen(
    [CLOUDFLARED, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

link = None
for riga in tunnel.stdout:
    m = re.search(r'https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com', riga)
    if m:
        link = m.group(0)
        print('\n==============================================')
        print('  LINK DA USARE CON genera.py (o nel browser):')
        print('  ' + link)
        print('==============================================\n')
        break

if link is None:
    comfy.terminate()
    raise RuntimeError('cloudflared non ha creato il link. Riesegui questa cella.')

# Continua a leggere l'output del tunnel per non bloccarlo
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print('Lascia questa cella in esecuzione. Per spegnere ComfyUI, interrompi la cella.')
try:
    while comfy.poll() is None:
        time.sleep(5)
    !tail -n 40 /tmp/comfyui.log
    print('ComfyUI si è fermato.')
finally:
    tunnel.terminate()
    comfy.terminate()